In [ ]:
# gt_pipeline_scaled.py
# Full Plan B pipeline scaled over all primitive terms.
# Each term gets its own folder under BASE_DIR.
# A diagnostic .txt is saved per term summarising bad months.

import os
import time
import random, math
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pytrends.request import TrendReq

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("C:/Python/CSUREMM/test")
START_DATE = pd.Timestamp("2022-01-01")
END_DATE   = pd.Timestamp("2026-05-31")
CHUNK_DAYS = 269
STEP_DAYS  = 240
SLEEP_SEC  = 8
GEO        = "US"
GPROP      = ""   # "" = web search only

PRIMITIVES = [
    "SUCCESS","SUCCESSFUL","TARIFF","THRIFT","THRIFTY","TREASURE",
    "UNDERWORLD","UNECONOMICAL","UNEMPLOYED","UNPROFITABLE","VAGABOND","VAGRANT",
    "VALUABLE","WARFARE","WASTE","WORTH",
]


# ── Helpers ───────────────────────────────────────────────────────────────────

def make_pytrends():
    return TrendReq(hl="en-US", tz=0, timeout=(10, 30), retries=3, backoff_factor=0.5)

def term_dir(term: str) -> Path:
    """Folder named after the lower-case term."""
    d = BASE_DIR / term.lower()
    d.mkdir(parents=True, exist_ok=True)
    return d

def window_filename(start, end, prefix):
    return f"{prefix}_{start.date()}_{end.date()}.csv"

def human_sleep(attempt: int, base: int = SLEEP_SEC):
    """
    Exponential backoff with full jitter.
    Attempt 1: 15–30s,  2: 15–60s,  3: 15–120s,  4: 15–240s,  5: 15–480s
    The random jitter means no two retries look the same to Google's detector.
    """
    cap   = base * (2 ** attempt)          # exponential ceiling: 30, 60, 120, 240, 480
    wait  = random.uniform(base, cap)      # random draw between floor and ceiling
    print(f"      sleeping {wait:.1f}s …")
    time.sleep(wait)

In [2]:
# ── Step 1: Download daily chunks ─────────────────────────────────────────────

def download_daily_chunks(term: str, folder: Path) -> list[str]:
    """
    Returns list of failed timeframes. Empty list = all chunks downloaded cleanly.
    """
    pytrends    = make_pytrends()
    chunk_start = START_DATE
    failed      = []

    while chunk_start <= END_DATE:
        chunk_end = min(chunk_start + pd.Timedelta(days=CHUNK_DAYS - 1), END_DATE)
        fname     = folder / window_filename(chunk_start, chunk_end, "gt_daily")

        if fname.exists():
            print(f"    [skip] {chunk_start.date()} → {chunk_end.date()} (already on disk)")
            chunk_start += pd.Timedelta(days=STEP_DAYS)
            continue

        timeframe = f"{chunk_start.date()} {chunk_end.date()}"
        saved     = False

        for attempt in range(1, 4):
            print(f"    [pull] {timeframe}  attempt {attempt}/3 …", end=" ", flush=True)
            try:
                pytrends.build_payload([term], cat=0, timeframe=timeframe, geo=GEO, gprop=GPROP)
                df = pytrends.interest_over_time()
                if df.empty or term not in df.columns:
                    print("empty — skipped")
                    break
                out = (df[[term]].reset_index()
                    .rename(columns={"date": "Time", term: "value"})[["Time", "value"]])
                out.to_csv(fname, index=False)
                print(f"{len(out)} rows → {fname.name}")
                saved = True
                break
            except Exception as e:
                if "429" in str(e):
                    print(f"429 —", end=" ")
                    human_sleep(attempt)
                    pytrends = make_pytrends()
                else:
                    print(f"ERROR: {e} — skipped")
                    break

        if not saved:
            print(f"    FAILED after 3 attempts: {timeframe}")
            failed.append(timeframe)

        chunk_start += pd.Timedelta(days=STEP_DAYS)
        time.sleep(random.uniform(8, 20))          # inter-chunk pause

    return failed                             # caller decides what to do


# ── Step 2: Download monthly anchor ───────────────────────────────────────────

def download_monthly_anchor(term: str, folder: Path) -> bool:
    """Returns True if anchor is available (existing or freshly downloaded)."""
    fname = folder / window_filename(START_DATE, END_DATE, "gt_monthly")
    if fname.exists():
        print(f"    [skip] monthly anchor (already on disk)")
        return True

    timeframe = f"{START_DATE.date()} {END_DATE.date()}"
    for attempt in range(1, 4):
        print(f"    [pull] monthly anchor {timeframe}  attempt {attempt}/3 …", end=" ", flush=True)
        try:
            pytrends = make_pytrends()
            pytrends.build_payload([term], cat=0, timeframe=timeframe, geo=GEO, gprop=GPROP)
            df = pytrends.interest_over_time()
            if df.empty or term not in df.columns:
                print("empty — not saved")
                return False
            out = (df[[term]].reset_index()
                   .rename(columns={"date": "Time", term: "monthly_gt"})[["Time", "monthly_gt"]])
            out.to_csv(fname, index=False)
            print(f"{len(out)} rows → {fname.name}")
            return True
        except Exception as e:
            if "429" in str(e):
                print(f"429 — waiting before retry …")
                time.sleep(random.uniform(8, 20)) 
            else:
                print(f"ERROR: {e} — skipped")
                return False

    print(f"    FAILED after 3 attempts: monthly anchor for '{term}'")
    return False




In [3]:
# ── Step 3: Classify files ─────────────────────────────────────────────────────

def classify_files(folder: Path):
    all_files    = list(folder.glob("*.csv"))
    daily_files  = []
    monthly_file = None

    for f in all_files:
        df         = pd.read_csv(f)
        dates      = pd.to_datetime(df["Time"])
        median_gap = dates.diff().dt.days.median()
        if median_gap <= 2:
            daily_files.append(f)
        else:
            monthly_file = f

    daily_files = sorted(daily_files, key=lambda f: pd.read_csv(f)["Time"].min())
    return daily_files, monthly_file



In [4]:
# ── Step 4: Stitch daily chunks ───────────────────────────────────────────────

def stitch_daily(daily_files: list) -> pd.DataFrame:
    dfs = []
    for f in daily_files:
        df = pd.read_csv(f, parse_dates=["Time"])
        df = df.rename(columns={c: "value" for c in df.columns if c != "Time"})
        df = df.sort_values("Time").drop_duplicates("Time")
        dfs.append(df)

    if not dfs:
        raise ValueError("No daily files to stitch.")

    stitched = dfs[0].copy()

    for i in range(1, len(dfs)):
        prev    = stitched
        curr    = dfs[i].copy()
        overlap = prev.merge(curr, on="Time", suffixes=("_prev", "_curr"))

        if len(overlap) >= 5:
            median_prev = overlap["value_prev"].median()
            median_curr = overlap["value_curr"].median()
            scale       = median_prev / median_curr if median_curr > 1e-9 else 1.0
        else:
            print(f"    WARNING: only {len(overlap)} overlap points for chunk {i} — scale=1.0")
            scale = 1.0

        curr         = curr.copy()
        curr["value"] *= scale
        combined     = pd.concat([prev, curr]).groupby("Time", as_index=False)["value"].mean()
        stitched     = combined.sort_values("Time").reset_index(drop=True)

    return stitched



In [5]:
# ── Step 5: Monthly anchor calibration (local per-month scale) ────────────────

def apply_monthly_anchor_local(stitched: pd.DataFrame, monthly_file) -> pd.DataFrame:
    monthly = pd.read_csv(monthly_file, parse_dates=["Time"]).set_index("Time").squeeze()
    monthly.name = "monthly"

    daily_agg = (stitched.set_index("Time")["value"]
                 .resample("MS").mean())
    daily_agg.name = "daily_agg"

    ratio = pd.concat([monthly, daily_agg], axis=1, sort=False).dropna()
    ratio = ratio[ratio["daily_agg"] > 1e-9]
    ratio["scale"] = ratio["monthly"] / ratio["daily_agg"]

    daily_scale = (ratio["scale"]
                   .reindex(pd.date_range(ratio.index.min(), ratio.index.max(), freq="D"))
                   .interpolate(method="time")
                   .ffill()
                   .bfill())

    result          = stitched.copy().set_index("Time")
    result["value"] = (result["value"] * daily_scale).clip(0, 100)
    return result.reset_index()


# ── Step 6: Save final series ─────────────────────────────────────────────────

def save_final(stitched: pd.DataFrame, term: str, folder: Path) -> pd.DataFrame:
    out   = stitched.rename(columns={"value": term.lower()})
    fname = folder / f"gt_stitched_{START_DATE.date()}_{END_DATE.date()}.csv"
    out.to_csv(fname, index=False)
    print(f"    Saved: {fname.name}  ({len(out)} rows)")
    return out


# ── Diagnostic txt ────────────────────────────────────────────────────────────

def save_diagnostic(stitched: pd.DataFrame, monthly_file, term: str, folder: Path):
    """
    Computes month-by-month ratio and writes a plain-text diagnostic report.
    Flags months outside [0.85, 1.15].
    """
    monthly   = pd.read_csv(monthly_file, parse_dates=["Time"]).set_index("Time").squeeze()
    daily_agg = (stitched.set_index("Time").iloc[:, 0]
                 .resample("MS").mean())
    daily_agg.name = "daily_agg"

    ratio = pd.concat([monthly, daily_agg], axis=1, sort=False).dropna()
    ratio.columns      = ["monthly", "daily_agg"]
    ratio["ratio"]     = ratio["daily_agg"] / ratio["monthly"].replace(0, pd.NA)
    bad                = ratio[ratio["ratio"].lt(0.85) | ratio["ratio"].gt(1.15)]

    lines = [
        f"Google Trends Diagnostic — term: {term.lower()}",
        f"Period : {START_DATE.date()} → {END_DATE.date()}",
        f"Rows in stitched series : {len(stitched)}",
        f"Monthly anchor points   : {len(ratio)}",
        f"",
        f"Months with ratio outside [0.85, 1.15]  ({len(bad)} found):",
        "",
    ]

    if bad.empty:
        lines.append("  None — alignment is clean.")
    else:
        lines.append(f"{'Time':<15} {'monthly':>10} {'daily_agg':>12} {'ratio':>8}")
        lines.append("-" * 50)
        for idx, row in bad.iterrows():
            lines.append(
                f"{str(idx.date()):<15} {row['monthly']:>10.1f} "
                f"{row['daily_agg']:>12.3f} {row['ratio']:>8.3f}"
            )

    lines += [
        "",
        "Full monthly comparison:",
        "",
        f"{'Time':<15} {'monthly':>10} {'daily_agg':>12} {'ratio':>8}",
        "-" * 50,
    ]
    for idx, row in ratio.iterrows():
        flag = "  ← !" if (row["ratio"] < 0.85 or row["ratio"] > 1.15) else ""
        lines.append(
            f"{str(idx.date()):<15} {row['monthly']:>10.1f} "
            f"{row['daily_agg']:>12.3f} {row['ratio']:>8.3f}{flag}"
        )

    txt_path = folder / "diagnostic.txt"
    txt_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"    Diagnostic saved: {txt_path.name}  ({len(bad)} bad months)")


In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────────────
# START_FROM = "liquidate"   # set to the term where 429 started, or None to run all

if __name__ == "__main__":
    total  = len(PRIMITIVES)
    errors = []

    # Skip terms before START_FROM
    primitives_to_run = PRIMITIVES
    if START_FROM:
        start_idx = next(
            (i for i, t in enumerate(PRIMITIVES) if t.lower() == START_FROM.lower()),
            0
        )
        primitives_to_run = PRIMITIVES[start_idx:]
        print(f"Resuming from '{START_FROM}' ({len(primitives_to_run)} terms remaining)")
    else:
        start_idx = 0

    try:                                             # ← outer try wraps the for loop
        for i, raw_term in enumerate(primitives_to_run, start_idx + 1):
            term   = raw_term.lower()
            folder = term_dir(term)

            print(f"\n{'='*60}")
            print(f"[{i}/{total}]  {term}")
            print(f"{'='*60}")

            try:                                     # ← inner try is inside the for loop
                print("  Step 1: downloading daily chunks …")
                failed_chunks = download_daily_chunks(term, folder)

                print("  Step 2: downloading monthly anchor …")
                monthly_ok = download_monthly_anchor(term, folder)

                if failed_chunks:
                    msg = (f"Skipping stitch — {len(failed_chunks)} chunk(s) missing:\n" +
                           "\n".join(f"  {c}" for c in failed_chunks))
                    print(f"  WARNING: {msg}")
                    (folder / "incomplete.txt").write_text(
                        f"term: {term}\n"
                        f"status: incomplete\n"
                        f"reason: {len(failed_chunks)} chunk(s) failed to download\n\n"
                        f"failed chunks:\n" +
                        "\n".join(failed_chunks),
                        encoding="utf-8"
                    )
                    errors.append((term, f"{len(failed_chunks)} chunk(s) failed"))
                    time.sleep(30)
                    continue                          # ← now correctly inside for loop

                print("  Step 3: classifying files …")
                daily_files, monthly_file = classify_files(folder)

                if not daily_files:
                    raise ValueError("No daily files found after download.")

                print("  Step 4: stitching daily chunks …")
                stitched = stitch_daily(daily_files)

                if monthly_ok and monthly_file:
                    print("  Step 5: applying local monthly anchor …")
                    stitched = apply_monthly_anchor_local(stitched, monthly_file)
                else:
                    print("  WARNING: no monthly anchor — skipping calibration.")

                print("  Step 6: saving final series …")
                final = save_final(stitched, term, folder)

                if monthly_ok and monthly_file:
                    print("  Saving diagnostic txt …")
                    save_diagnostic(final, monthly_file, term, folder)

                incomplete = folder / "incomplete.txt"
                if incomplete.exists():
                    incomplete.unlink()

            except KeyboardInterrupt:
                raise                                # ← bubble up to outer handler

            except Exception as e:
                print(f"  ERROR on term '{term}': {e}")
                errors.append((term, str(e)))

            time.sleep(random.uniform(20, 45))       # inter-term cooldown

    except KeyboardInterrupt:
        print(f"\n  Last attempted term : {term}")
        print(f"  Set START_FROM = '{term}' to resume from here.")

    finally:
        print(f"\n{'='*60}")
        print(f"Pipeline stopped.")

Resuming from 'liquidate' (63 terms remaining)

[91/153]  liquidate
  Step 1: downloading daily chunks …
    [skip] 2022-01-01 → 2022-09-26 (already on disk)
    [pull] 2022-08-29 2023-05-24  attempt 1/3 … 269 rows → gt_daily_2022-08-29_2023-05-24.csv
    [pull] 2023-04-26 2024-01-19  attempt 1/3 … 269 rows → gt_daily_2023-04-26_2024-01-19.csv
    [pull] 2023-12-22 2024-09-15  attempt 1/3 … 269 rows → gt_daily_2023-12-22_2024-09-15.csv
    [pull] 2024-08-18 2025-05-13  attempt 1/3 … 429 —       sleeping 15.4s …
    [pull] 2024-08-18 2025-05-13  attempt 2/3 … 269 rows → gt_daily_2024-08-18_2025-05-13.csv
    [pull] 2025-04-15 2026-01-08  attempt 1/3 … 269 rows → gt_daily_2025-04-15_2026-01-08.csv
    [pull] 2025-12-11 2026-05-31  attempt 1/3 … 172 rows → gt_daily_2025-12-11_2026-05-31.csv
  Step 2: downloading monthly anchor …
    [pull] monthly anchor 2022-01-01 2026-05-31  attempt 1/3 … 232 rows → gt_monthly_2022-01-01_2026-05-31.csv
  Step 3: classifying files …
  Step 4: stitching d